In [3]:
from typing import TypedDict,Literal
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义状态
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type:str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"poem": response.content}

def node_b(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")])
    return {"joke": response.content}

def route(state: OverAllState) ->Literal["a", "b"]:
    if "诗" in state["content_type"]:
        return "a"
    else:
        return "b"

# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_conditional_edges(START,route,path_map={
    "a": "node_a",
    "b": "node_b"
})
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()
graph.invoke({"topic": "猫猫", "content_type": "诗"})

{'topic': '猫猫',
 'poem': '《戏咏猫》\n玉爪金瞳夜不眠，花阴扑影自相怜。\n偶逢蝶过墙头戏，却逐鼠逃檐角旋。\n饱食自知鱼味美，安眠偏爱日高眠。\n狸奴纵有擒贼技，只傍书窗伴我闲。\n\n注：此诗以唐代咏物诗风格，刻画猫之形态与灵性。首联“玉爪金瞳”状其形貌，“花阴扑影”写其嬉戏之态。颔联对仗工整，以蝶鼠之戏衬其矫健。颈联转写猫之慵懒，与尾联“伴我闲”呼应，赋予猫以文人雅士般的闲适意趣。诗中“狸奴”为古人对猫的雅称，符合唐诗用语习惯。',
 'content_type': '诗'}